In [5]:
import os
import time
from dotenv import load_dotenv

from langchain_openai import OpenAIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.embeddings import OllamaEmbeddings

from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores import Chroma

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [6]:
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

print("OpenAI key loaded:", api_key is not None)

OpenAI key loaded: True


In [7]:
docs = []

for file in os.listdir("data"):
    loader = TextLoader(os.path.join("data", file))
    docs.extend(loader.load())

print("Documents loaded:", len(docs))

Documents loaded: 5


In [8]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

chunks = splitter.split_documents(docs)

print("Total chunks:", len(chunks))
print("Example chunk:\n", chunks[0].page_content)

Total chunks: 12
Example chunk:
 Artificial Intelligence (AI) refers to the simulation of human intelligence in machines.

AI systems can learn from data, recognize patterns, and make decisions.


In [9]:
openai_embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vector = openai_embeddings.embed_query(chunks[0].page_content)

print("Embedding vector length:", len(vector))
print("Sample embedding values:", vector[:5])

Embedding vector length: 1536
Sample embedding values: [-0.021480116993188858, -0.001280980883166194, -0.015780305489897728, 0.011114140041172504, 0.04894552752375603]


### My Observations

I used the OpenAI embedding model "text-embedding-3-small" to convert text
chunks into vector representations.

The embedding vector length returned was 1536 dimensions.

Each value in the vector is a floating-point number representing the semantic
meaning of the text.

I observed that OpenAI embeddings are generated very quickly using the API.
However, they require internet connectivity and API credits.

In [10]:
hf_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_hf = hf_embeddings.embed_query(chunks[0].page_content)

print("HF embedding length:", len(vector_hf))
print("Sample values:", vector_hf[:5])

C:\Users\kirut\AppData\Local\Temp\ipykernel_15640\3585054065.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  hf_embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1060.87it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HF embedding length: 384
Sample values: [-0.028206419199705124, -0.018697643652558327, -0.012520058080554008, 0.01596260257065296, -0.003790842369198799]


### Observations

The HuggingFace embedding model runs locally and does not require an API key.

The embedding vector dimension is 384 for the MiniLM model.

It is slower than OpenAI embeddings when running on CPU,
but it is useful for offline workflows.

### Comparison

OpenAI embeddings are cloud-based and provide high quality semantic
representations with larger embedding dimensions.

HuggingFace embeddings run locally and are free to use.

OpenAI is better for production systems where accuracy matters,
while HuggingFace is useful for local experimentation and privacy-sensitive data.

In [11]:
embeddings = [openai_embeddings.embed_query(c.page_content) for c in chunks]

def search(query, top_k=3):

    query_vector = openai_embeddings.embed_query(query)

    scores = []

    for i, vec in enumerate(embeddings):

        score = sum(a*b for a,b in zip(vec,query_vector))
        scores.append((score,i))

    scores.sort(reverse=True)

    for score, idx in scores[:top_k]:
        print("\nScore:", score)
        print(chunks[idx].page_content)

In [12]:
search("What is machine learning?")
search("Explain vector databases")
search("What are large language models?")


Score: 0.6152770616232808
Machine Learning is a subset of Artificial Intelligence.

It focuses on building algorithms that learn patterns from data.

Score: 0.5000209996633331
Machine learning is widely used in recommendation systems,
image recognition, and predictive analytics.

Score: 0.4637845235699836
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines.

AI systems can learn from data, recognize patterns, and make decisions.

Score: 0.6380647589034835
Vector databases store numerical embeddings representing text, images, or other data.

Embeddings capture semantic meaning so similar concepts are close in vector space.

Score: 0.5992931707171533
Vector databases enable similarity search using metrics like cosine similarity.

Popular vector databases include FAISS, ChromaDB, Pinecone, and Weaviate.

Score: 0.42126916513525975
Retrieval Augmented Generation (RAG) combines information retrieval with language models.

Instead of relying only on model

In [13]:
vectorstore = FAISS.from_documents(
    chunks,
    openai_embeddings
)

results = vectorstore.similarity_search(
    "Explain RAG", k=3
)

for r in results:
    print("\n", r.page_content)


 Retrieval Augmented Generation (RAG) combines information retrieval with language models.

Instead of relying only on model training data, RAG retrieves relevant documents from
a vector database.

 These documents are provided as context to the LLM during generation.

RAG improves factual accuracy and reduces hallucinations in generative AI systems.

 LLMs can perform tasks such as question answering, summarization,
translation, and code generation.

They are the foundation of modern generative AI applications.


In [14]:
ollama_embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

vector_ollama = ollama_embeddings.embed_query(
    chunks[0].page_content
)

print("Ollama embedding length:", len(vector_ollama))

C:\Users\kirut\AppData\Local\Temp\ipykernel_15640\1656173204.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  ollama_embeddings = OllamaEmbeddings(


Ollama embedding length: 768


In [15]:
start=time.time()
openai_embeddings.embed_query(chunks[0].page_content)
print("OpenAI time:", time.time()-start)

start=time.time()
hf_embeddings.embed_query(chunks[0].page_content)
print("HF time:", time.time()-start)

start=time.time()
ollama_embeddings.embed_query(chunks[0].page_content)
print("Ollama time:", time.time()-start)

OpenAI time: 1.1188669204711914
HF time: 0.025995492935180664
Ollama time: 2.1675000190734863


In [16]:
faiss_store = FAISS.from_documents(
    chunks,
    hf_embeddings
)

faiss_store.save_local("faiss_index")

In [17]:
db = FAISS.load_local(
    "faiss_index",
    hf_embeddings,
    allow_dangerous_deserialization=True
)

db.similarity_search("machine learning",k=2)

[Document(id='1300647b-b40d-4e84-bc4c-8521cbda17bd', metadata={'source': 'data\\ml.txt'}, page_content='Machine Learning is a subset of Artificial Intelligence.\n\nIt focuses on building algorithms that learn patterns from data.'),
 Document(id='bdd09204-6df2-4e4b-8f29-2b07c1cd1f29', metadata={'source': 'data\\ml.txt'}, page_content='Machine learning is widely used in recommendation systems,\nimage recognition, and predictive analytics.')]

In [18]:
chroma_db = Chroma.from_documents(
    chunks,
    hf_embeddings,
    persist_directory="chroma_db"
)

results = chroma_db.similarity_search("AI applications")

for r in results:
    print(r.page_content)

Applications include healthcare diagnosis, autonomous driving, finance fraud detection,
robotics, and natural language processing.
AI is a broad field including machine learning, deep learning, and reinforcement learning.
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines.

AI systems can learn from data, recognize patterns, and make decisions.
Machine Learning is a subset of Artificial Intelligence.

It focuses on building algorithms that learn patterns from data.


FAISS is optimized for fast in-memory similarity search.

ChromaDB supports persistent storage and easy integration with LangChain.

FAISS is suitable for high-performance search systems,
while ChromaDB is useful when embeddings need to be stored on disk.

In [19]:
def build_pipeline(embedding_type="openai"):

    if embedding_type=="openai":
        emb=openai_embeddings

    elif embedding_type=="hf":
        emb=hf_embeddings

    elif embedding_type=="ollama":
        emb=ollama_embeddings

    store=FAISS.from_documents(chunks,emb)

    return store

In [20]:
pipeline=build_pipeline("hf")

results=pipeline.similarity_search("What is AI?")

for r in results:
    print(r.page_content)

Artificial Intelligence (AI) refers to the simulation of human intelligence in machines.

AI systems can learn from data, recognize patterns, and make decisions.
AI is a broad field including machine learning, deep learning, and reinforcement learning.
Machine Learning is a subset of Artificial Intelligence.

It focuses on building algorithms that learn patterns from data.
Applications include healthcare diagnosis, autonomous driving, finance fraud detection,
robotics, and natural language processing.


Embeddings are important because they convert text into numerical vectors
that capture semantic meaning.

Vector databases are required because traditional databases cannot perform
efficient similarity search on high-dimensional vectors.

This pipeline forms the foundation of RAG systems because it allows
relevant documents to be retrieved before generating answers with an LLM.